# Module 2: Fixed Effects Implementation

## Learning Objectives

By completing this notebook, you will:
1. Implement fixed effects using the within transformation from scratch
2. Apply fixed effects using linearmodels (Python) and plm (R)
3. Compare LSDV (Least Squares Dummy Variables) to within estimation
4. Interpret fixed effects coefficients correctly
5. Compute and extract entity fixed effects

## Prerequisites

- Module 0 and 1 completed
- Understanding of omitted variable bias
- Panel data structure (MultiIndex)

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Panel data regression
from linearmodels.panel import PanelOLS

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

## 1. The Fixed Effects Model

### Model Specification

$$y_{it} = \alpha_i + X_{it}\beta + \epsilon_{it}$$

where:
- $\alpha_i$ is the **entity fixed effect** (time-invariant)
- $X_{it}$ are time-varying predictors
- $\beta$ is the **within effect** we want to estimate

### Key Assumption

**Strict exogeneity (conditional on $\alpha_i$):**

$$E[\epsilon_{it} | X_{i1}, ..., X_{iT}, \alpha_i] = 0$$

This allows $Cov(\alpha_i, X_{it}) \neq 0$ (correlation between fixed effect and predictors).

### Why This Matters

If unobserved entity characteristics $\alpha_i$ correlate with predictors, pooled OLS is **biased**. Fixed effects solves this by eliminating $\alpha_i$.

## 2. Generate Panel Data with Entity Effects

Let's create data where pooled OLS will be biased but fixed effects is consistent.

In [ ]:
# Data generating process
n_firms = 100
n_years = 10

# Create panel structure
firms = np.arange(1, n_firms + 1)
years = np.arange(2010, 2010 + n_years)

panel_index = pd.MultiIndex.from_product(
    [firms, years],
    names=['firm_id', 'year']
)

# Entity fixed effects (firm productivity)
# These represent unobserved firm quality
alpha_i = np.random.randn(n_firms) * 30 + 50

# Generate R&D spending (correlated with firm quality)
# High-quality firms invest more in R&D
rd_spending = []
for i in range(n_firms):
    # Base R&D depends on firm quality (alpha_i)
    base_rd = 10 + 0.3 * alpha_i[i] + np.random.randn() * 5
    
    # R&D varies over time
    rd_i = [base_rd]
    for t in range(1, n_years):
        # AR(1) process with mean reversion
        rd_i.append(0.6 * rd_i[-1] + 0.4 * base_rd + np.random.randn() * 3)
    
    rd_spending.extend(rd_i)

rd_spending = np.array(rd_spending)

# Generate patents (true effect: beta = 2.0)
alpha_repeated = np.repeat(alpha_i, n_years)
epsilon = np.random.randn(n_firms * n_years) * 8

# True model: patents = alpha_i + 2.0 * R&D + error
beta_true = 2.0
patents = alpha_repeated + beta_true * rd_spending + epsilon

# Create DataFrame
df = pd.DataFrame({
    'patents': patents,
    'rd_spending': rd_spending
}, index=panel_index).reset_index()

print(f"Generated panel data:")
print(f"  Firms: {n_firms}")
print(f"  Years: {n_years}")
print(f"  Total observations: {len(df)}")
print(f"  True causal effect (beta): {beta_true}")
print(f"\nFirst 15 rows:")
print(df.head(15))

### Visualize the Data

Let's plot a few firms to see within and between variation.

In [ ]:
# Plot 6 random firms
sample_firms = np.random.choice(firms, 6, replace=False)

fig, ax = plt.subplots(figsize=(12, 6))

for firm in sample_firms:
    firm_data = df[df['firm_id'] == firm]
    ax.plot(firm_data['year'], firm_data['patents'], 
            marker='o', label=f'Firm {firm}', alpha=0.7)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Patents', fontsize=12)
ax.set_title('Patent Production Over Time (Sample Firms)', fontsize=14)
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Firms differ in their baseline patent production (between variation)")
print("      and show changes over time (within variation)")

## 3. Pooled OLS: Biased Estimator

First, let's see what happens if we ignore the panel structure and estimate pooled OLS.

In [ ]:
# Pooled OLS: Ignores entity effects
X_pooled = sm.add_constant(df['rd_spending'])
y_pooled = df['patents']

model_pooled = sm.OLS(y_pooled, X_pooled)
results_pooled = model_pooled.fit()

print("POOLED OLS Results")
print("=" * 70)
print(results_pooled.summary().tables[1])
print("\n" + "=" * 70)
print(f"True beta: {beta_true:.4f}")
print(f"Pooled OLS beta: {results_pooled.params['rd_spending']:.4f}")
print(f"Bias: {results_pooled.params['rd_spending'] - beta_true:.4f}")
print("\n⚠️  Pooled OLS is BIASED upward due to omitted variable (firm quality)")

### Why is Pooled OLS Biased?

Because:
1. High-quality firms ($\alpha_i$ large) produce more patents
2. High-quality firms also spend more on R&D
3. Pooled OLS attributes the effect of firm quality to R&D spending

This is **omitted variable bias**:

$$\text{Bias} = \text{Cov}(\alpha_i, X_{it}) \times \text{Effect of } \alpha_i \text{ on } y$$

## 4. Fixed Effects: Within Transformation (From Scratch)

### The Within Transformation

**Step 1:** Compute entity means

$$\bar{y}_i = \frac{1}{T_i} \sum_{t=1}^{T_i} y_{it}$$
$$\bar{X}_i = \frac{1}{T_i} \sum_{t=1}^{T_i} X_{it}$$

**Step 2:** Demean (subtract entity means)

$$\tilde{y}_{it} = y_{it} - \bar{y}_i$$
$$\tilde{X}_{it} = X_{it} - \bar{X}_i$$

**Step 3:** Run OLS on demeaned data

$$\tilde{y}_{it} = \tilde{X}_{it}\beta + \tilde{\epsilon}_{it}$$

The $\alpha_i$ terms cancel out!

In [ ]:
def fixed_effects_within(df, entity_col, y_col, X_cols):
    """
    Estimate fixed effects using within transformation.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
        Entity identifier
    y_col : str
        Dependent variable
    X_cols : list
        Independent variables
    
    Returns
    -------
    dict
        Estimation results including coefficients, SEs, fixed effects
    """
    # Step 1: Compute entity means
    entity_means_y = df.groupby(entity_col)[y_col].transform('mean')
    entity_means_X = df.groupby(entity_col)[X_cols].transform('mean')
    
    # Step 2: Demean
    y_demeaned = df[y_col] - entity_means_y
    X_demeaned = df[X_cols] - entity_means_X
    
    # Step 3: OLS on demeaned data (no intercept needed)
    model_within = sm.OLS(y_demeaned, X_demeaned)
    results_within = model_within.fit()
    
    # Recover fixed effects: alpha_i = y_bar_i - X_bar_i * beta
    entity_means_y_unique = df.groupby(entity_col)[y_col].mean()
    entity_means_X_unique = df.groupby(entity_col)[X_cols].mean()
    
    # Predicted means from model
    if len(X_cols) == 1:
        predicted_means = entity_means_X_unique * results_within.params[0]
    else:
        predicted_means = (entity_means_X_unique * results_within.params).sum(axis=1)
    
    fixed_effects = entity_means_y_unique - predicted_means
    
    # Overall intercept (mean of fixed effects)
    overall_intercept = fixed_effects.mean()
    
    return {
        'beta': results_within.params,
        'se': results_within.bse,
        't_stats': results_within.tvalues,
        'p_values': results_within.pvalues,
        'r_squared_within': results_within.rsquared,
        'fixed_effects': fixed_effects,
        'overall_intercept': overall_intercept,
        'results_object': results_within
    }

# Estimate fixed effects
fe_results = fixed_effects_within(df, 'firm_id', 'patents', ['rd_spending'])

print("FIXED EFFECTS (Within Transformation) Results")
print("=" * 70)
print(f"{'Variable':<15} {'Coef':>10} {'Std Err':>10} {'t':>8} {'P>|t|':>10}")
print("=" * 70)
print(f"{'rd_spending':<15} {fe_results['beta']['rd_spending']:>10.4f} "
      f"{fe_results['se']['rd_spending']:>10.4f} "
      f"{fe_results['t_stats']['rd_spending']:>8.3f} "
      f"{fe_results['p_values']['rd_spending']:>10.4f}")
print("=" * 70)
print(f"R-squared (within): {fe_results['r_squared_within']:.4f}")
print(f"Overall intercept: {fe_results['overall_intercept']:.4f}")
print("\n" + "=" * 70)
print(f"True beta: {beta_true:.4f}")
print(f"Fixed effects beta: {fe_results['beta']['rd_spending']:.4f}")
print(f"Bias: {fe_results['beta']['rd_spending'] - beta_true:.4f}")
print("\n✅ Fixed effects is UNBIASED (estimates true causal effect)")

### Extract and Examine Fixed Effects

In [ ]:
# Compare estimated fixed effects to true values
fe_comparison = pd.DataFrame({
    'firm_id': np.arange(1, n_firms + 1),
    'true_alpha': alpha_i,
    'estimated_alpha': fe_results['fixed_effects'].values
})

# Add overall intercept to estimated alphas for comparison
fe_comparison['estimated_alpha_centered'] = (
    fe_comparison['estimated_alpha'] + fe_results['overall_intercept']
)

print("Fixed Effects: True vs Estimated (first 10 firms)")
print(fe_comparison.head(10))

# Plot correlation
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(fe_comparison['true_alpha'], 
           fe_comparison['estimated_alpha_centered'],
           alpha=0.5, s=30)

# Add 45-degree line
min_val = min(fe_comparison['true_alpha'].min(), 
              fe_comparison['estimated_alpha_centered'].min())
max_val = max(fe_comparison['true_alpha'].max(), 
              fe_comparison['estimated_alpha_centered'].max())
ax.plot([min_val, max_val], [min_val, max_val], 
        'r--', linewidth=2, label='45° line')

# Correlation
corr = fe_comparison[['true_alpha', 'estimated_alpha_centered']].corr().iloc[0, 1]
ax.text(0.05, 0.95, f'Correlation: {corr:.3f}',
        transform=ax.transAxes, fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('True Fixed Effect', fontsize=12)
ax.set_ylabel('Estimated Fixed Effect', fontsize=12)
ax.set_title('Fixed Effects: True vs Estimated', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Fixed Effects Using linearmodels

The `linearmodels` package provides efficient implementation of panel models.

In [ ]:
# Set multi-index for linearmodels
df_panel = df.set_index(['firm_id', 'year'])

# Entity fixed effects
model_fe = PanelOLS(
    dependent=df_panel['patents'],
    exog=df_panel[['rd_spending']],
    entity_effects=True  # Include entity fixed effects
)

# Fit with clustered standard errors (cluster by firm)
results_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)

print("FIXED EFFECTS (linearmodels) Results")
print("=" * 70)
print(results_fe)

# Compare to our manual implementation
print("\n" + "=" * 70)
print("Comparison: Manual vs linearmodels")
print("=" * 70)
print(f"Manual within:     {fe_results['beta']['rd_spending']:.6f}")
print(f"linearmodels:      {results_fe.params['rd_spending']:.6f}")
print(f"Difference:        {abs(fe_results['beta']['rd_spending'] - results_fe.params['rd_spending']):.2e}")
print("\n✅ Implementations match!")

## 6. LSDV: Least Squares Dummy Variables

An alternative to within transformation is to include dummy variables for each entity.

$$y_{it} = \sum_{i=1}^n \alpha_i D_i + X_{it}\beta + \epsilon_{it}$$

where $D_i = 1$ if observation is from entity $i$, 0 otherwise.

**This gives identical $\beta$ estimates as within transformation.**

In [ ]:
# LSDV approach (WARNING: Slow for large N, creates many dummies)
# Only estimate on subset for demonstration
df_subset = df[df['firm_id'] <= 20].copy()

# Create entity dummies using pd.get_dummies
firm_dummies = pd.get_dummies(df_subset['firm_id'], prefix='firm', drop_first=True)

# Combine with predictor
X_lsdv = pd.concat([df_subset[['rd_spending']], firm_dummies], axis=1)
X_lsdv = sm.add_constant(X_lsdv)
y_lsdv = df_subset['patents']

# Estimate
model_lsdv = sm.OLS(y_lsdv, X_lsdv)
results_lsdv = model_lsdv.fit()

print("LSDV Results (subset of 20 firms)")
print("=" * 70)
print("Coefficient on rd_spending:")
print(f"  Estimate: {results_lsdv.params['rd_spending']:.6f}")
print(f"  Std Error: {results_lsdv.bse['rd_spending']:.6f}")
print(f"  t-stat: {results_lsdv.tvalues['rd_spending']:.3f}")
print(f"  p-value: {results_lsdv.pvalues['rd_spending']:.4f}")

# Compare to within estimator on same subset
fe_subset = fixed_effects_within(df_subset, 'firm_id', 'patents', ['rd_spending'])
print("\nComparison: LSDV vs Within (same subset)")
print("=" * 70)
print(f"LSDV:    {results_lsdv.params['rd_spending']:.6f}")
print(f"Within:  {fe_subset['beta']['rd_spending']:.6f}")
print(f"Difference: {abs(results_lsdv.params['rd_spending'] - fe_subset['beta']['rd_spending']):.2e}")
print("\n✅ LSDV and within transformation give identical beta estimates!")

### When to Use LSDV vs Within

**Use Within Transformation:**
- Large number of entities (N > 100)
- More computationally efficient
- Standard approach in econometrics

**Use LSDV:**
- Small number of entities
- Want to see individual fixed effects directly
- Testing specific hypotheses about entity effects

## 7. Interpretation of Fixed Effects

### What Does the Coefficient Mean?

The FE coefficient represents the **within-entity** effect:

> "For a given firm, a 1-unit increase in R&D spending is associated with a {beta}-unit increase in patents, controlling for all time-invariant firm characteristics."

### What You Cannot Conclude

- Cannot compare **between** firms
- Cannot estimate effects of time-invariant characteristics
- Interpretation is **conditional on entity**

In [ ]:
# Demonstrate interpretation visually
sample_firm = 1
firm_data = df[df['firm_id'] == sample_firm].copy()

# Fit within-firm regression
X_firm = sm.add_constant(firm_data['rd_spending'])
y_firm = firm_data['patents']
model_firm = sm.OLS(y_firm, X_firm).fit()

fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot
ax.scatter(firm_data['rd_spending'], firm_data['patents'], 
           s=100, alpha=0.6, label='Observations')

# Fitted line
X_line = np.linspace(firm_data['rd_spending'].min(), 
                     firm_data['rd_spending'].max(), 100)
y_line = model_firm.params['const'] + model_firm.params['rd_spending'] * X_line
ax.plot(X_line, y_line, 'r-', linewidth=2, 
        label=f'Within-firm slope: {model_firm.params["rd_spending"]:.2f}')

ax.set_xlabel('R&D Spending', fontsize=12)
ax.set_ylabel('Patents', fontsize=12)
ax.set_title(f'Within-Firm Relationship (Firm {sample_firm})', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Interpretation for Firm {sample_firm}:")
print(f"  When this firm increases R&D by 1 unit, patents increase by {model_firm.params['rd_spending']:.2f}")
print(f"\nFixed effects averages this within-firm slope across all firms.")
print(f"FE estimate: {fe_results['beta']['rd_spending']:.2f}")

---

## Exercises

### Exercise 1: Implement First Differences Estimator

**Task:** The **first differences** (FD) estimator is an alternative to within transformation. It differences adjacent time periods:

$$\Delta y_{it} = y_{it} - y_{i,t-1} = \Delta X_{it} \beta + \Delta \epsilon_{it}$$

Implement FD and compare to FE.

**Hints:**
- Use `groupby().diff()` to create differences
- Run OLS on differenced data (no intercept)
- Drop first period (NaN after differencing)

In [ ]:
# YOUR CODE HERE
def first_differences(df, entity_col, y_col, X_cols):
    """
    Estimate using first differences.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        Results with beta, se, etc.
    """
    # TODO: Implement first differences estimator
    pass

In [ ]:
# SOLUTION (hidden)
def first_differences_solution(df, entity_col, y_col, X_cols):
    """Estimate using first differences."""
    df_sorted = df.sort_values([entity_col, 'year'])
    
    # Create differences
    y_diff = df_sorted.groupby(entity_col)[y_col].diff()
    X_diff = df_sorted.groupby(entity_col)[X_cols].diff()
    
    # Drop NaN (first period for each entity)
    valid = ~y_diff.isna()
    y_diff_clean = y_diff[valid]
    X_diff_clean = X_diff[valid]
    
    # OLS on differences (no intercept)
    model_fd = sm.OLS(y_diff_clean, X_diff_clean)
    results_fd = model_fd.fit()
    
    return {
        'beta': results_fd.params,
        'se': results_fd.bse,
        't_stats': results_fd.tvalues,
        'p_values': results_fd.pvalues,
        'r_squared': results_fd.rsquared
    }

In [ ]:
# Test your function
def test_exercise_1():
    """Test first differences implementation."""
    fd_results = first_differences(df, 'firm_id', 'patents', ['rd_spending'])
    
    assert fd_results is not None, "Function returns None - did you implement it?"
    assert 'beta' in fd_results, "Missing 'beta' in results"
    
    # FD and FE should be similar (but not identical)
    fd_beta = fd_results['beta']['rd_spending']
    fe_beta = fe_results['beta']['rd_spending']
    
    # Check it's in reasonable range
    assert 1.5 < fd_beta < 2.5, f"FD estimate {fd_beta:.4f} seems unreasonable"
    
    print("✅ Correct! First differences estimator implemented.")
    print("\nComparison:")
    print(f"  Fixed Effects: {fe_beta:.4f}")
    print(f"  First Differences: {fd_beta:.4f}")
    print(f"  True beta: {beta_true:.4f}")
    print("\nNote: FE and FD should be similar but not identical.")
    print("      FE is more efficient when errors are serially uncorrelated.")
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: F-Test for Entity Fixed Effects

**Task:** Test whether entity fixed effects are jointly significant.

**Null hypothesis:** $H_0: \alpha_1 = \alpha_2 = ... = \alpha_n$

**F-statistic:**

$$F = \frac{(SSR_{pooled} - SSR_{FE}) / (n-1)}{SSR_{FE} / (nT - n - k)}$$

where $n$ = number of entities, $T$ = time periods, $k$ = number of predictors.

**Hints:**
- Compute SSR from pooled OLS and FE
- Use `stats.f.cdf()` for p-value

In [ ]:
# YOUR CODE HERE
def f_test_entity_effects(df, entity_col, y_col, X_cols):
    """
    F-test for significance of entity fixed effects.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        F-statistic and p-value
    """
    # TODO: Implement F-test
    pass

In [ ]:
# SOLUTION (hidden)
def f_test_entity_effects_solution(df, entity_col, y_col, X_cols):
    """F-test for significance of entity fixed effects."""
    # Pooled OLS
    X_pool = sm.add_constant(df[X_cols])
    y_pool = df[y_col]
    pooled = sm.OLS(y_pool, X_pool).fit()
    ssr_pooled = pooled.ssr
    
    # Fixed effects
    fe = fixed_effects_within(df, entity_col, y_col, X_cols)
    ssr_fe = fe['results_object'].ssr
    
    # Degrees of freedom
    n = df[entity_col].nunique()
    T = df.groupby(entity_col).size().mean()
    k = len(X_cols)
    
    df1 = n - 1
    df2 = n * T - n - k
    
    # F-statistic
    f_stat = ((ssr_pooled - ssr_fe) / df1) / (ssr_fe / df2)
    
    # P-value
    p_value = 1 - stats.f.cdf(f_stat, df1, df2)
    
    return {
        'f_statistic': f_stat,
        'df1': df1,
        'df2': df2,
        'p_value': p_value
    }

In [ ]:
# Test your function
def test_exercise_2():
    """Test F-test for entity effects."""
    f_test_results = f_test_entity_effects(df, 'firm_id', 'patents', ['rd_spending'])
    
    assert f_test_results is not None, "Function returns None - did you implement it?"
    assert 'f_statistic' in f_test_results, "Missing 'f_statistic'"
    assert 'p_value' in f_test_results, "Missing 'p_value'"
    
    # F-stat should be large (fixed effects are important)
    assert f_test_results['f_statistic'] > 1, "F-statistic should be > 1"
    
    print("✅ Correct! F-test for entity fixed effects:")
    print("=" * 50)
    print(f"F-statistic: {f_test_results['f_statistic']:.2f}")
    print(f"Degrees of freedom: ({f_test_results['df1']}, {f_test_results['df2']})")
    print(f"P-value: {f_test_results['p_value']:.4e}")
    
    if f_test_results['p_value'] < 0.01:
        print("\n✅ Reject null: Entity fixed effects are significant!")
    else:
        print("\n⚠️  Cannot reject null: Entity effects may not be significant")
    
    return True

# Uncomment to test
# test_exercise_2()

### Exercise 3: Between Estimator

**Task:** The **between estimator** uses only cross-sectional variation (entity means).

$$\bar{y}_i = \alpha + \bar{X}_i\beta_{between} + \bar{\epsilon}_i$$

Implement the between estimator and compare to FE.

**Hints:**
- Compute entity means for y and X
- Run OLS on these means
- Should be biased (uses between variation)

In [ ]:
# YOUR CODE HERE
def between_estimator(df, entity_col, y_col, X_cols):
    """
    Estimate using between (entity means) variation.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        Results
    """
    # TODO: Implement between estimator
    pass

In [ ]:
# SOLUTION (hidden)
def between_estimator_solution(df, entity_col, y_col, X_cols):
    """Estimate using between (entity means) variation."""
    # Compute entity means
    entity_means = df.groupby(entity_col)[[y_col] + X_cols].mean().reset_index()
    
    # OLS on means
    X_between = sm.add_constant(entity_means[X_cols])
    y_between = entity_means[y_col]
    
    model_between = sm.OLS(y_between, X_between)
    results_between = model_between.fit()
    
    return {
        'beta': results_between.params,
        'se': results_between.bse,
        't_stats': results_between.tvalues,
        'p_values': results_between.pvalues,
        'r_squared': results_between.rsquared
    }

In [ ]:
# Test your function
def test_exercise_3():
    """Test between estimator."""
    between_results = between_estimator(df, 'firm_id', 'patents', ['rd_spending'])
    
    assert between_results is not None, "Function returns None - did you implement it?"
    assert 'beta' in between_results, "Missing 'beta' in results"
    
    between_beta = between_results['beta']['rd_spending']
    
    print("✅ Correct! Between estimator implemented.")
    print("\nComparison of Estimators:")
    print("=" * 50)
    print(f"{'Estimator':<20} {'Beta':>12} {'Uses'}")
    print("=" * 50)
    print(f"{'True effect':<20} {beta_true:>12.4f} {''}")
    print(f"{'Pooled OLS':<20} {results_pooled.params['rd_spending']:>12.4f} {'Both (biased)'}")
    print(f"{'Fixed Effects':<20} {fe_results['beta']['rd_spending']:>12.4f} {'Within only'}")
    print(f"{'Between':<20} {between_beta:>12.4f} {'Between only'}")
    print("\nNote: Between estimator is biased (like pooled OLS)")
    print("      because it uses cross-sectional variation.")
    return True

# Uncomment to test
# test_exercise_3()

---

## Summary

### Key Takeaways

1. **Fixed Effects Eliminates Bias:** By demeaning data, FE removes time-invariant confounders

2. **Within Transformation:** Subtract entity means to eliminate $\alpha_i$

3. **LSDV Equivalence:** Including entity dummies gives same $\beta$ as within transformation

4. **Interpretation:** FE coefficient is the **within-entity** effect, not between-entity comparison

5. **Cannot Estimate Time-Invariant Effects:** Variables that don't change within entity are absorbed by fixed effects

### When to Use Fixed Effects

✅ **Use FE when:**
- Entity-specific confounders likely present
- Correlation between $\alpha_i$ and $X_{it}$ suspected
- Sufficient within-entity variation in X
- Interest in causal within-entity effects

❌ **Do NOT use FE when:**
- X is time-invariant (gender, industry)
- Little within-entity variation in X
- Between-entity comparisons are the goal
- Random effects assumptions clearly hold

### What's Next

In the next notebook, we'll:
- Test FE assumptions
- Diagnose specification issues
- Handle clustered standard errors
- Perform robustness checks

---

**Next:** `02_fe_diagnostics.ipynb` - Fixed Effects Diagnostics and Testing